# L8: Lab - Robot Memory Agent

In this lab, you'll build an on-device memory system for a robot. The robot observes its environment, stores observations as embeddings in Qdrant Edge, and uses that memory to make navigation decisions.

This is a simplified version of how physical robots build world models for obstacle avoidance and spatial awareness. Real robotics platforms (RB3, RB5, Snapdragon Ride) use this same pattern running on Snapdragon chipsets.

```
Sensors --> Embedding Model (AI Hub) --> Qdrant Edge (spatial memory)
                                              |
                                     Query: "Is this area safe?"
                                              |
                                     Navigation Decision
```

## Setup

In [ ]:
!pip install qdrant-edge-py qai-hub "qai-hub-models[nomic_embed_text]" torch transformers matplotlib numpy

In [ ]:
import os
import sys
sys.path.append("..")

import torch
import numpy as np
import time
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from transformers import AutoTokenizer
from qai_hub_models.models.nomic_embed_text import Model as NomicEmbedText
from qdrant_edge import (
    EdgeShard, EdgeConfig, VectorDataConfig, Distance,
    Point, UpdateOperation, Query, QueryRequest,
)

# Load embedding model
text_model = NomicEmbedText.from_pretrained()
tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5")

EMBEDDING_DIM = 512
MAX_SEQ_LEN = 512
VECTOR_NAME = "observation"

def embed(texts, prefix="search_document: "):
    """Generate embeddings using the nomic model."""
    prefixed = [prefix + t for t in texts]
    encoded = tokenizer(
        prefixed, padding="max_length", truncation=True,
        max_length=MAX_SEQ_LEN, return_tensors="pt",
    )
    with torch.no_grad():
        return text_model(encoded["input_ids"], encoded["attention_mask"])

print(f"Embedding model loaded: nomic_embed_text ({EMBEDDING_DIM}d)")

### Compile for Snapdragon

In [ ]:
import qai_hub
from utils import get_ai_hub_api_token, get_random_device

ai_hub_api_token = get_ai_hub_api_token()
!qai-hub configure --api_token $ai_hub_api_token

device_name = get_random_device()
device = qai_hub.Device(device_name)

example_ids = torch.randint(0, tokenizer.vocab_size, (1, MAX_SEQ_LEN))
example_mask = torch.ones(1, MAX_SEQ_LEN, dtype=torch.long)
traced_model = torch.jit.trace(text_model, (example_ids, example_mask))

compile_job = qai_hub.submit_compile_job(
    model=traced_model,
    input_specs={"input_ids": (1, MAX_SEQ_LEN), "attention_mask": (1, MAX_SEQ_LEN)},
    device=device,
)
target_model = compile_job.get_target_model()
print(f"Model compiled for {device_name}")

## 1. Define the Robot's World

A 16x16 grid with rooms, corridors, obstacles, and hazard zones. This is closer to a real warehouse or office floor plan than a tiny 8x8 toy grid.

In [ ]:
GRID_SIZE = 16

# 0 = free, 1 = obstacle, 2 = hazard
world = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)

# Room walls (simulating an office/warehouse layout)
# Vertical wall segments
for r in range(2, 7):
    world[r, 5] = 1
for r in range(9, 14):
    world[r, 5] = 1
for r in range(2, 7):
    world[r, 10] = 1
for r in range(9, 14):
    world[r, 10] = 1

# Horizontal wall segments
for c in range(2, 5):
    world[7, c] = 1
for c in range(6, 10):
    world[7, c] = 1
for c in range(11, 14):
    world[7, c] = 1

# Scattered obstacles (furniture, equipment)
obstacle_positions = [
    (1, 2), (1, 8), (1, 13),
    (3, 3), (3, 7), (3, 12),
    (5, 1), (5, 8), (5, 13),
    (10, 2), (10, 7), (10, 13),
    (12, 3), (12, 8), (12, 12),
    (14, 1), (14, 6), (14, 14),
]
for r, c in obstacle_positions:
    world[r, c] = 1

# Hazard zones (wet floor, restricted area, high-voltage)
hazard_positions = [
    (13, 0), (13, 1), (14, 0),  # bottom-left hazard zone
    (0, 14), (0, 15), (1, 15),  # top-right hazard zone
    (8, 8), (8, 9), (9, 8),     # center hazard zone
    (15, 12), (15, 13),          # bottom-right hazard
]
for r, c in hazard_positions:
    world[r, c] = 2

# Count cells
n_free = np.sum(world == 0)
n_obstacle = np.sum(world == 1)
n_hazard = np.sum(world == 2)
print(f"World: {GRID_SIZE}x{GRID_SIZE} grid")
print(f"  Free: {n_free}, Obstacles: {n_obstacle}, Hazards: {n_hazard}")

def visualize_world(robot_pos=None, path=None, title=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    color_map = {0: "white", 1: "gray", 2: "#FF6B6B"}

    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            color = color_map[world[r, c]]
            ax.add_patch(patches.Rectangle((c, GRID_SIZE-1-r), 1, 1,
                         fill=True, facecolor=color, edgecolor="black", linewidth=0.5))

    if path:
        for i, (r, c) in enumerate(path):
            alpha = 0.3 + 0.7 * (i / len(path))
            ax.plot(c + 0.5, GRID_SIZE - 1 - r + 0.5, "b.", markersize=6, alpha=alpha)

    if robot_pos:
        r, c = robot_pos
        ax.plot(c + 0.5, GRID_SIZE - 1 - r + 0.5, "go", markersize=12)

    ax.set_xlim(0, GRID_SIZE)
    ax.set_ylim(0, GRID_SIZE)
    ax.set_aspect("equal")
    ax.set_title(title or "Robot World (green=robot, gray=obstacle, red=hazard)")
    plt.show()

visualize_world(robot_pos=(0, 0))

## 2. Create the Robot's Memory

The robot stores **observations** as text descriptions embedded into vectors. Each observation captures what the robot "sees" at a position.

In [ ]:
SHARD_DIR = "./robot_memory_shard"
Path(SHARD_DIR).mkdir(parents=True, exist_ok=True)

config = EdgeConfig(
    vector_data={
        VECTOR_NAME: VectorDataConfig(
            size=EMBEDDING_DIM,
            distance=Distance.Cosine,
        )
    }
)

memory = EdgeShard(SHARD_DIR, config)
print("Robot memory initialized")

## 3. Observation Function

The robot looks at its surroundings and generates a text description. In a real system, this would come from camera/LIDAR sensors processed by a vision model compiled via AI Hub.

In [ ]:
observation_counter = 0

def observe(row, col):
    """Generate an observation description based on what the robot sees."""
    cell = world[row, col]

    if cell == 0:
        current = "open space, safe to move through"
    elif cell == 1:
        current = "solid obstacle blocking the path, cannot pass"
    else:
        current = "dangerous hazard area, must avoid"

    neighbors = []
    for dr, dc, direction in [(-1,0,"north"), (1,0,"south"), (0,-1,"west"), (0,1,"east")]:
        nr, nc = row + dr, col + dc
        if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE:
            ncell = world[nr, nc]
            if ncell == 1:
                neighbors.append(f"obstacle to the {direction}")
            elif ncell == 2:
                neighbors.append(f"hazard to the {direction}")
            else:
                neighbors.append(f"clear path to the {direction}")

    description = f"Position ({row},{col}): {current}. " + ", ".join(neighbors)
    return description

def store_observation(row, col):
    """Observe and store in memory."""
    global observation_counter
    desc = observe(row, col)
    embedding = embed([desc])

    point = Point(
        id=observation_counter,
        vector={VECTOR_NAME: embedding[0].tolist()},
        payload={
            "description": desc,
            "row": row,
            "col": col,
            "cell_type": int(world[row, col]),
            "timestamp": time.time(),
            "safe": world[row, col] == 0,
        }
    )
    memory.update(UpdateOperation.upsert_points([point]))
    observation_counter += 1
    return desc

# Test
desc = observe(0, 0)
print(f"Sample observation: {desc}")

## 4. Exploration Phase: Multiple Patrol Routes

The robot explores the world in three patrol routes, building up spatial memory over time. Each route covers a different section of the facility.

In [ ]:
# Patrol route 1: Top half perimeter and corridors
route1 = (
    [(0, c) for c in range(16)] +           # Top edge left to right
    [(r, 15) for r in range(1, 8)] +         # Right side down
    [(r, 6) for r in range(0, 7)] +          # Through corridor between rooms
    [(r, 11) for r in range(0, 7)] +         # Through second corridor
    [(1, c) for c in range(0, 16)]            # Row 1 sweep
)

# Patrol route 2: Bottom half
route2 = (
    [(15, c) for c in range(16)] +           # Bottom edge
    [(r, 0) for r in range(15, 7, -1)] +     # Left side up
    [(r, 15) for r in range(8, 16)] +        # Right side down
    [(r, 6) for r in range(8, 16)] +         # Corridor
    [(r, 11) for r in range(8, 16)] +        # Second corridor
    [(9, c) for c in range(0, 16)]            # Row 9 sweep
)

# Patrol route 3: Interior sweep
route3 = (
    [(r, c) for r in range(4, 12, 2) for c in range(1, 15, 2)] +  # Grid sample
    [(8, c) for c in range(0, 16)] +          # Middle row
    [(r, 8) for r in range(0, 16)]            # Middle column
)

# Filter out positions that are obstacles (robot can't stand there)
def filter_valid(route):
    seen = set()
    valid = []
    for r, c in route:
        if 0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE and (r, c) not in seen:
            seen.add((r, c))
            valid.append((r, c))
    return valid

route1 = filter_valid(route1)
route2 = filter_valid(route2)
route3 = filter_valid(route3)
all_explored = set()

print("Patrol Route 1 (top half)...")
for row, col in route1:
    store_observation(row, col)
    all_explored.add((row, col))
print(f"  Explored {len(route1)} positions, memory: {observation_counter}")

print("Patrol Route 2 (bottom half)...")
for row, col in route2:
    store_observation(row, col)
    all_explored.add((row, col))
print(f"  Explored {len(route2)} positions, memory: {observation_counter}")

print("Patrol Route 3 (interior sweep)...")
for row, col in route3:
    if (row, col) not in all_explored:  # Skip already observed
        store_observation(row, col)
        all_explored.add((row, col))
print(f"  Explored {len([p for p in route3 if p in all_explored])} positions, memory: {observation_counter}")

print(f"\nTotal: {observation_counter} observations stored")
print(f"Unique positions explored: {len(all_explored)} / {GRID_SIZE * GRID_SIZE}")

visualize_world(
    robot_pos=route3[-1],
    path=list(all_explored),
    title=f"Explored {len(all_explored)} positions across 3 patrol routes"
)

## 5. Memory-Based Navigation Queries

The robot can query its memory to make decisions. Instead of re-exploring, it uses past observations.

In [ ]:
def query_memory(question, limit=3):
    """Ask the robot's memory a question."""
    query_emb = embed([question], prefix="search_query: ")
    results = memory.query(
        QueryRequest(
            query=Query.Nearest(query_emb[0].tolist(), using=VECTOR_NAME),
            limit=limit,
            with_vector=False,
            with_payload=True,
        )
    )
    return results

queries = [
    "Where are the obstacles?",
    "Which areas are dangerous?",
    "Where can I move safely?",
    "Is there a clear path to the east?",
]

for q in queries:
    print(f"\nQ: {q}")
    results = query_memory(q)
    for r in results:
        p = r.payload
        safe_str = "SAFE" if p["safe"] else "UNSAFE"
        print(f"  [{r.score:.3f}] ({p['row']},{p['col']}) [{safe_str}] {p['description'][:80]}...")

## 6. Safety-Filtered Navigation

Use payload filters to only retrieve safe positions when planning a route.

In [ ]:
def find_safe_positions_near(description, limit=5):
    """Find explored positions that are safe and match a description."""
    query_emb = embed([description], prefix="search_query: ")
    results = memory.query(
        QueryRequest(
            query=Query.Nearest(query_emb[0].tolist(), using=VECTOR_NAME),
            limit=limit,
            with_vector=False,
            with_payload=True,
            filter={"must": [{"key": "safe", "match": {"value": True}}]}
        )
    )
    return results

print("Safe positions near 'open area with clear paths':")
results = find_safe_positions_near("open area with clear paths in all directions")
safe_positions = []
for r in results:
    p = r.payload
    pos = (p["row"], p["col"])
    safe_positions.append(pos)
    print(f"  ({p['row']},{p['col']}) score={r.score:.3f}")

visualize_world(robot_pos=exploration_path[-1], path=safe_positions)

## 7. Memory Decay: Forget Old Observations

On a resource-constrained robot (4GB RAM on RB3), you can't store everything forever. Implement a decay policy that keeps only the most recent N observations, evicting old ones by timestamp. This maps to the memory decay discussion from the team meeting in L5.

In [ ]:
all_points = memory.retrieve(
    point_ids=list(range(observation_counter)),
    with_payload=True,
    with_vector=False,
)
print(f"Current memory: {len(all_points)} observations")

# Memory cap: keep only the most recent 80 observations (simulating constrained device)
MEMORY_CAP = 80
sorted_by_time = sorted(all_points, key=lambda p: p.payload["timestamp"], reverse=True)
keep = sorted_by_time[:MEMORY_CAP]
evict = sorted_by_time[MEMORY_CAP:]

print(f"Memory cap: {MEMORY_CAP}")
print(f"Would keep: {len(keep)} most recent observations")
print(f"Would evict: {len(evict)} oldest observations")

if evict:
    evict_ids = [p.id for p in evict]
    print(f"Evicting {len(evict_ids)} old observations...")
    # In production: memory.update(UpdateOperation.delete_points(evict_ids))
    print(f"(Skipping actual deletion for this demo)")
    print(f"Oldest kept: position ({keep[-1].payload['row']},{keep[-1].payload['col']})")
    print(f"Newest kept: position ({keep[0].payload['row']},{keep[0].payload['col']})")

## 8. Cleanup

In [ ]:
memory.close()

import shutil
shutil.rmtree(SHARD_DIR, ignore_errors=True)
print("Cleaned up")

## Summary

In this lab you built a robot memory agent that:
- Uses `nomic_embed_text` from `qai_hub_models`, compiled for Snapdragon via AI Hub
- Navigates a 16x16 grid (256 cells) with rooms, corridors, obstacles, and hazard zones
- Executes 3 patrol routes, storing 100+ spatial observations as embeddings
- Queries past observations to answer navigation questions ("Where are the obstacles?")
- Filters memory by safety status for route planning
- Implements memory decay with a cap of 80 observations for resource-constrained devices (4GB RAM on RB3)

This pattern applies to any agent that needs to remember and reason about its physical environment: delivery robots, warehouse automation, autonomous vehicles, and more.